In [ ]:
from pathlib import Path
import sys
import importlib
import json
import os
import time
import math
from collections.abc import Mapping

import numpy as np
import pandas as pd

_CWD = Path.cwd().resolve()
if (_CWD / "HFSS_Horn").is_dir():
    _REPO_DIR = _CWD
    _HORN_DIR = _CWD / "HFSS_Horn"
else:
    _HORN_DIR = _CWD
    _REPO_DIR = _CWD.parent

for _path in (_REPO_DIR, _HORN_DIR):
    _path_str = str(_path)
    if _path.exists() and _path_str not in sys.path:
        sys.path.insert(0, _path_str)

import lib_config as config
import lib_backbone
import lib_gp

lib_backbone = importlib.reload(lib_backbone)

%load_ext autoreload
%autoreload 2


In [ ]:
config_path = _HORN_DIR / "_config.toml"
if not config_path.exists():
    raise FileNotFoundError(f"HFSS Horn config not found: {config_path}")

_config = config._loadConfig(config_path)
app_config = config.initParams(_config, debug=True)

backbone = lib_backbone.Backbone(config=app_config)
base_dir = app_config.env.dir_base
cfg = app_config.hfss
ROUND_DECIMALS = app_config.runtime.round_decimals

LOWER_BOUNDS = np.asarray(cfg.lower_bounds, dtype=float)
UPPER_BOUNDS = np.asarray(cfg.upper_bounds, dtype=float)
BOUNDS = np.vstack([LOWER_BOUNDS, UPPER_BOUNDS]).T
PARAM_NAMES = list(cfg.param_names)
OBJECTIVE_COL = app_config.objective.name

print("n_params:", cfg.n_params)
print("PARAM_NAMES:", PARAM_NAMES)
print("ROUND_DECIMALS:", ROUND_DECIMALS)


In [ ]:
x_center = np.asarray(getattr(cfg, "baseline", [6.4, 20.0, 0.18, 0.27, 0.21, 0.20, 0.14]), dtype=float)

N_VALIDATION = 100
RANDOM_SEED = 101
RUN_ALL_VALIDATION_ROWS = True
DEBUG_N_ROWS = 5
CUSTOM_PERTURBATION_STD = {
    "d_m": 0.1,
    "l_tot": 0.1,
    "f_wg": 0.01,
    "f_t1": 0.01,
    "f_mid": 0.01,
    "f_t2": 0.01,
    "f_ap": 0.01,
}
PERTURBATION_ENABLED = {name: True for name in PARAM_NAMES}
CUSTOM_PERTURBATION_STD_ARRAY = np.asarray(
    [CUSTOM_PERTURBATION_STD[name] if PERTURBATION_ENABLED[name] else 0.0 for name in PARAM_NAMES],
    dtype=float,
)
PERTURBATION_SIGMA = np.diag(CUSTOM_PERTURBATION_STD_ARRAY**2)
FIXED_PARAM_NAMES = [name for name in PARAM_NAMES if not PERTURBATION_ENABLED[name]]


In [ ]:
SECTION_FRAC_SLICE = slice(2, 7)
SECTION_FRAC_TOL = 1.0e-8


def section_frac_sum_constraint(x):
    values = np.asarray(x, dtype=float).flatten()[SECTION_FRAC_SLICE]
    return float(np.sum(values) - 1.0)


def enforce_horn_constraints(params, decimals=ROUND_DECIMALS):
    return lib_backbone.enforce_section_frac_constraint(
        params,
        lower_bounds=LOWER_BOUNDS,
        upper_bounds=UPPER_BOUNDS,
        decimals=decimals,
    )


def assert_horn_constraints(params, tol=SECTION_FRAC_TOL):
    residual = section_frac_sum_constraint(params)
    if abs(residual) > tol:
        raise ValueError(f"section_fracs must sum to 1.0; residual={residual}")
    return True


def validate_x_center(x, n_params, lower_bounds, upper_bounds):
    x = np.asarray(x, dtype=float).reshape(-1)
    if x.size != n_params:
        raise ValueError(f"x_center must have length cfg.n_params={n_params}, but got length {x.size}.")
    x = enforce_horn_constraints(x)
    below = x < lower_bounds
    above = x > upper_bounds
    if np.any(below) or np.any(above):
        bad = [(PARAM_NAMES[i], float(x[i]), float(lower_bounds[i]), float(upper_bounds[i])) for i in np.where(below | above)[0]]
        raise ValueError(f"x_center has out-of-bounds values: {bad}")
    assert_horn_constraints(x)
    return np.round(x, ROUND_DECIMALS)


def build_validation_sigma(custom_sigma):
    sigma = np.asarray(custom_sigma, dtype=float)
    expected_shape = (cfg.n_params, cfg.n_params)
    if sigma.shape != expected_shape:
        raise ValueError(f"Sigma must have shape {expected_shape}, got {sigma.shape}.")
    return sigma


In [ ]:
x_center = validate_x_center(x_center, cfg.n_params, LOWER_BOUNDS, UPPER_BOUNDS)
VALIDATION_SIGMA = build_validation_sigma(PERTURBATION_SIGMA)
rng = np.random.default_rng(RANDOM_SEED)

Z_perturbation = lib_gp.sample_input_perturbations(
    x=x_center,
    Sigma=VALIDATION_SIGMA,
    n_samples=N_VALIDATION,
    bounds=BOUNDS,
    rng=rng,
)
Z_perturbation = np.asarray([enforce_horn_constraints(row) for row in Z_perturbation], dtype=float)

if FIXED_PARAM_NAMES:
    fixed_indices = [PARAM_NAMES.index(name) for name in FIXED_PARAM_NAMES]
    Z_perturbation[:, fixed_indices] = x_center[fixed_indices]
    Z_perturbation = np.asarray([enforce_horn_constraints(row) for row in Z_perturbation], dtype=float)

Z_validation_with_center = np.vstack([x_center, Z_perturbation])
parametric_input_df = pd.DataFrame(Z_validation_with_center, columns=PARAM_NAMES)
parametric_input_df.index.name = "sample_idx"

print("Fixed parameters:", FIXED_PARAM_NAMES if FIXED_PARAM_NAMES else "None")
print("DataFrame shape:", parametric_input_df.shape)
display(parametric_input_df.head())


In [ ]:
backbone.mkdir()
run_dir = backbone._get_dir_run()
parametric_input_csv = run_dir / "parametric_input.csv"
parametric_input_df.to_csv(parametric_input_csv, index=False)
print(f"Saved parametric input CSV: {parametric_input_csv}")
print(f"Rows x columns: {parametric_input_df.shape[0]} x {parametric_input_df.shape[1]}")


In [ ]:
backbone.initStorer(mode="w")
run_dir = backbone._get_dir_run()
model_paths, model_paths_str = backbone._get_path_models()
RESULTS_FILE = str(run_dir / Path(_config["io"]["filename_output"]))
LEGACY_TEMP_FILE = str(run_dir / Path(_config["io"].get("filename_temp", "temp_hfss_export.csv")))
TEMP_OUTPUTS = [{"name": output.name, "path": run_dir / Path(output.filename)} for output in app_config.io.temp_outputs]
if not TEMP_OUTPUTS:
    TEMP_OUTPUTS = [{"name": "S11", "path": Path(LEGACY_TEMP_FILE)}]

print("Run directory:", run_dir)
print("RESULTS_FILE:", RESULTS_FILE)
print("TEMP_OUTPUTS:", TEMP_OUTPUTS)


In [ ]:
def _ensure_finite(name, value, quantity_name=None):
    label = f" for physical quantity '{quantity_name}'" if quantity_name else ""
    value = float(value)
    if not math.isfinite(value):
        raise ValueError(f"{name}{label} must be finite; got {value!r}")
    return value


def normalize_objective(value, target, limit):
    value = _ensure_finite("value", value)
    target = _ensure_finite("target", target)
    limit = _ensure_finite("limit", limit)
    if target == limit:
        raise ValueError(f"target and limit must differ; got target={target}, limit={limit}")
    return max(0.0, (value - target) / (limit - target))


def _objective_terms_to_mapping(objective_config):
    terms = objective_config.get("terms") if isinstance(objective_config, Mapping) else getattr(objective_config, "terms", None)
    if terms is None:
        raise ValueError("objective configuration must contain 'terms'")
    term_map = {}
    for term in terms:
        column = term.get("column") if isinstance(term, Mapping) else getattr(term, "column", None)
        if not column:
            raise ValueError("each objective term must define a non-empty 'column'")
        if column in term_map:
            raise ValueError(f"duplicate objective term for physical quantity '{column}'")
        term_map[column] = term
    return term_map


def _term_value(term, key, quantity_name):
    if isinstance(term, Mapping):
        if key not in term:
            raise ValueError(f"objective term for physical quantity '{quantity_name}' is missing '{key}'")
        return term[key]
    if not hasattr(term, key):
        raise ValueError(f"objective term for physical quantity '{quantity_name}' is missing '{key}'")
    return getattr(term, key)


def calculate_lp_fom(values, objective_config, p=2.0):
    p = _ensure_finite("p", p)
    if p < 1.0:
        raise ValueError(f"p must be >= 1; got {p}")
    term_map = _objective_terms_to_mapping(objective_config)
    missing_values = sorted(set(term_map) - set(values))
    missing_config = sorted(set(values) - set(term_map))
    if missing_values:
        raise ValueError(f"values are missing configured physical quantities: {missing_values}")
    if missing_config:
        raise ValueError(f"objective configuration is missing physical quantities: {missing_config}")
    weighted_sum = 0.0
    weight_sum = 0.0
    for quantity_name in sorted(term_map):
        term = term_map[quantity_name]
        value = _ensure_finite("value", values[quantity_name], quantity_name)
        target = _ensure_finite("target", _term_value(term, "target", quantity_name), quantity_name)
        limit = _ensure_finite("limit", _term_value(term, "limit", quantity_name), quantity_name)
        weight = _ensure_finite("weight", _term_value(term, "weight", quantity_name), quantity_name)
        if weight < 0.0:
            raise ValueError(f"weight for physical quantity '{quantity_name}' must be non-negative; got {weight}")
        weighted_sum += weight * normalize_objective(value, target, limit) ** p
        weight_sum += weight
    if weight_sum <= 0.0:
        raise ValueError("sum of objective weights must be greater than 0")
    return (weighted_sum / weight_sum) ** (1.0 / p)


In [ ]:
def wait_for_temp_outputs(temp_outputs, timeout_sec=600, poll_interval_sec=1.0):
    start = time.time()
    paths = [Path(item["path"]) for item in temp_outputs]
    while True:
        if all(path.exists() for path in paths):
            return
        if time.time() - start > timeout_sec:
            raise TimeoutError(f"Timed out waiting for temp outputs: {paths}")
        time.sleep(poll_interval_sec)


def read_temp_output_like_current_s11(temp_hfss_path):
    df_temp = pd.read_csv(temp_hfss_path)
    return df_temp.iloc[-1, -1]


def read_all_temp_outputs(temp_outputs):
    return {item["name"]: read_temp_output_like_current_s11(item["path"]) for item in temp_outputs}


def compute_objective(outputs, objective_cfg):
    return calculate_lp_fom(outputs, objective_cfg, p=getattr(objective_cfg, "p", 2.0))


def round_numeric_row(row, decimals=ROUND_DECIMALS):
    rounded = dict(row)
    for key, value in list(rounded.items()):
        if pd.notna(value) and isinstance(value, (int, float, np.integer, np.floating)):
            rounded[key] = float(np.round(value, decimals))
    return rounded


def getResult(input_params, param_names, temp_outputs, result_file_path):
    wait_for_temp_outputs(temp_outputs)
    header_flag = not os.path.exists(result_file_path)
    outputs = read_all_temp_outputs(temp_outputs)
    result_row = dict(zip(param_names, input_params))
    result_row.update(outputs)
    result_row[OBJECTIVE_COL] = compute_objective(outputs, app_config.objective)
    result_row = round_numeric_row(result_row)
    pd.DataFrame([result_row]).to_csv(result_file_path, mode="a", header=header_flag, index=False)
    for item in temp_outputs:
        try:
            os.remove(item["path"])
        except OSError:
            pass
    return result_row


def target_hfss(_config_temp, sim_id, param_names, params):
    params = enforce_horn_constraints(params)
    assert_horn_constraints(params)
    backbone.call_subroutine(_config_temp, sim_id, param_names, params, value_fmt=f"{{:.{ROUND_DECIMALS}f}}")
    return getResult(params, param_names, TEMP_OUTPUTS, RESULTS_FILE)


def costFunctionWrapper(param_names, params):
    params = enforce_horn_constraints(params)
    sim_id = backbone._getSimulationID()
    result_row = target_hfss(_config_temp, sim_id, param_names, params)
    return float(result_row[OBJECTIVE_COL]), round_numeric_row(result_row)


In [ ]:
rows_to_run = parametric_input_df if RUN_ALL_VALIDATION_ROWS else parametric_input_df.iloc[:DEBUG_N_ROWS]
_config_temp = {
    "n_simulation": int(len(rows_to_run)),
    "n_repeats": 1,
    "WATCH_DIR": str(run_dir),
    "INPUT_FILE": str(run_dir / Path(_config["io"]["filename_input"])),
    "MODEL_FILE": model_paths_str,
    "RESULTS_FILE": RESULTS_FILE,
    "TEMP_FILE": LEGACY_TEMP_FILE,
    "TEMP_OUTPUTS": [{"name": item["name"], "path": str(item["path"])} for item in TEMP_OUTPUTS],
    "DONE_FLAG_FILE": str(run_dir / Path("hfss.done")),
}
done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
done_flag_path.unlink(missing_ok=True)
with open(base_dir / Path("_config_HFSS.json"), "w") as f:
    json.dump(_config_temp, f, indent=4)
print(f"Temporarily updated '{base_dir / Path('_config_HFSS.json')}' with run-specific WATCH_DIR for HFSS.")
print("Number of validation simulations:", len(rows_to_run))


In [ ]:
parametric_history = []
start = time.perf_counter()

try:
    for sample_idx, (_, row) in enumerate(rows_to_run.iterrows(), start=1):
        params = row[PARAM_NAMES].to_numpy(dtype=float)
        print(f"[Horn validation] HFSS evaluation {sample_idx}/{len(rows_to_run)}")
        _, result_row = costFunctionWrapper(PARAM_NAMES, params)
        result_row["sample_idx"] = int(sample_idx - 1)
        parametric_history.append(result_row)

    df_final = pd.DataFrame(parametric_history)
    df_final[PARAM_NAMES] = df_final[PARAM_NAMES].round(ROUND_DECIMALS)
    for column in df_final.select_dtypes(include=[np.number]).columns:
        df_final[column] = df_final[column].round(ROUND_DECIMALS)

    df_output = backbone._genOutputDataFrame(df_final)
    if "best" in df_output:
        df_output["best"] = df_output["best"].round(ROUND_DECIMALS)

    parametric_results_csv = run_dir / "parametric_results.csv"
    df_output.to_csv(parametric_results_csv, index=False)
    backbone._addNewDatasetToHDF(df_output.select_dtypes(include=[np.number]), "output", "repeat_1")

    elapsed = time.perf_counter() - start
    best_idx = df_output[OBJECTIVE_COL].idxmin()
    print("-" * 75)
    print(f"Horn validation finished in {elapsed:.3f} seconds")
    print(f"Best {OBJECTIVE_COL}: {df_output.loc[best_idx, OBJECTIVE_COL]:.10f}")
    display(df_output.loc[[best_idx]])
    print(f"Saved results CSV: {parametric_results_csv}")
    print(f"Saved HDF5: {run_dir / 'results.h5'}")

finally:
    done_flag_path.touch()
    json_file = base_dir / Path("_config_HFSS.json")
    json_file.unlink(missing_ok=True)
    if backbone.h5f:
        backbone.h5f.close()


In [ ]:
def compute_mc_susceptibility(results_df, x_center, param_names, sigma_vec, objective_col=OBJECTIVE_COL, center_row_index=0, ridge=1e-10):
    x_center = np.asarray(x_center, dtype=float).reshape(-1)
    sigma_vec = np.asarray(sigma_vec, dtype=float).reshape(-1)
    df = results_df.copy()
    center_mask = df["sample_idx"].to_numpy(dtype=int) == int(center_row_index) if "sample_idx" in df else df.index.to_numpy() == int(center_row_index)
    if not np.any(center_mask):
        raise ValueError(f"Center row index {center_row_index} not found in results.")
    y_center = float(df.loc[center_mask, objective_col].iloc[0])
    sample_df = df.loc[~center_mask].copy()
    X_delta = sample_df[param_names].to_numpy(dtype=float) - x_center
    y_delta = sample_df[objective_col].to_numpy(dtype=float) - y_center
    XtX = X_delta.T @ X_delta
    beta = np.linalg.solve(XtX + ridge * np.eye(XtX.shape[0]), X_delta.T @ y_delta)
    contribution = (sigma_vec * beta) ** 2
    total = float(np.sum(contribution))
    susceptibility_df = pd.DataFrame(
        {
            "param": param_names,
            "sigma": sigma_vec,
            "beta": beta,
            "sigma_abs_beta": np.abs(sigma_vec * beta),
            "contribution": contribution,
            "contribution_ratio": contribution / total if total > 0 else np.nan,
        }
    ).sort_values("contribution", ascending=False)
    return susceptibility_df, {"center_objective": y_center, "total_susceptibility": total, "n_perturbation_rows": int(len(sample_df))}


results_for_susceptibility = pd.DataFrame(parametric_history) if "parametric_history" in globals() and parametric_history else pd.read_csv(run_dir / "parametric_results.csv")
susceptibility_df, susceptibility_summary = compute_mc_susceptibility(
    results_for_susceptibility,
    x_center=x_center,
    param_names=PARAM_NAMES,
    sigma_vec=CUSTOM_PERTURBATION_STD_ARRAY,
)
susceptibility_csv = run_dir / "mc_susceptibility.csv"
susceptibility_df.to_csv(susceptibility_csv, index=False)
print(susceptibility_summary)
print(f"Saved susceptibility CSV: {susceptibility_csv}")
display(susceptibility_df)
